In [ ]:
%load_ext autoreload
%autoreload 2
from girder_client import GirderClient
from dotenv import load_dotenv
import os
import json
import pandas as pd
from augment import get_igsn_xrd_links, extract_alpss_from_portal, download_csv_to_numpy
import io

# --- Load environment variables from .env ---
load_dotenv()

GIRDER_API_URL = os.getenv("GIRDER_API_URL")
API_KEY = os.getenv("HTMDEC_GIRDER_API_KEY")
ALPSS_RESULTS_FOLDER = "692da3649ca0e7d06ecd3d61"
ALPSS_FORM_ID = '6931adb5bb4e23b1e86a1ff1'

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [17]:
client = GirderClient(apiUrl=GIRDER_API_URL)
client.authenticate(apiKey=API_KEY)
user = client.get("user/me")
print(f"✅ Authenticated as {user['login']}")

✅ Authenticated as arachid1


# XRD

In [18]:
igsn_xrd_links = get_igsn_xrd_links(igsn='JHAMAD00001', client=client)
print(igsn_xrd_links)

['https://data.htmdec.org/#item/692ef750a27c05f33c327d34', 'https://data.htmdec.org/#item/692ef793bb4e23b1e86a1edd', 'https://data.htmdec.org/#item/692ef7dabb4e23b1e86a1ee9', 'https://data.htmdec.org/#item/692ef817a27c05f33c327d4c', 'https://data.htmdec.org/#item/692ef85aa27c05f33c327d52', 'https://data.htmdec.org/#item/692ef93aa27c05f33c327d5b', 'https://data.htmdec.org/#item/692ef93d6edf207f7858129a', 'https://data.htmdec.org/#item/692ef97d6edf207f785812a3', 'https://data.htmdec.org/#item/692ef9bba27c05f33c327d76', 'https://data.htmdec.org/#item/692ef9fdbfb7a20ff4e7d4fe', 'https://data.htmdec.org/#item/692efbdf6edf207f785812bf', 'https://data.htmdec.org/#item/692efbe2bb4e23b1e86a1f0b', 'https://data.htmdec.org/#item/692efc1d6edf207f785812c5', 'https://data.htmdec.org/#item/692efc5fa27c05f33c327d89', 'https://data.htmdec.org/#item/692efc7ea27c05f33c327d92', 'https://data.htmdec.org/#item/69303b56bfb7a20ff4e7d53f', 'https://data.htmdec.org/#item/69303b59bfb7a20ff4e7d545', 'https://data

In [19]:
igsn_xrd_csv = download_csv_to_numpy(client, igsn_xrd_links[0])
print(igsn_xrd_csv.shape)

(1000, 2)


# ALPSS

In [21]:
velocity_time_history, flyer_velocity, spall_strength = extract_alpss_from_portal('C1--20250605--00087', ALPSS_FORM_ID, ALPSS_RESULTS_FOLDER, client)
print(flyer_velocity)
print(spall_strength)

905.7013045502234
19152542751.358006


In [23]:
igsn_velocity_history_csv = download_csv_to_numpy(client, velocity_time_history)
print(igsn_velocity_history_csv.shape)

(19199, 2)


# Curation Example 

In [3]:
path = "test.xlsx"
df = pd.read_excel(path)

In [6]:
df["Sample microstructure/material characterization (EBSD/XRD images)"] = df["Sample_ID"].apply(
    lambda x: get_igsn_xrd_links(x, client)
)

In [ ]:
df[[
    "velocity-time history",
    "Flyer velocity",
    "spall strength"
]] = df['PDV_FileName'].apply(
    lambda fn: extract_alpss_from_portal(fn, ALPSS_FORM_ID, ALPSS_RESULTS_FOLDER, client)
).apply(pd.Series)

In [9]:
df.to_excel("augmented_output.xlsx", index=False)